# Ch 30. ZeRO and FSDP — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Calculate ZeRO Stage 0/1/2/3 memory for a 70B model with N=16.

### Solution

A simplified mixed-precision Adam model uses 2P bytes for FP16 parameters, 2P for gradients, and 12P for FP32 master weights plus moments: 16P total. Stages 1/2/3 shard optimizer, then gradients, then parameters across $N$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
P=70; N=16
mem={'stage0':16*P,'stage1':4*P+12*P/N,'stage2':4*P+(2+12)*P/N,'stage3':16*P/N}
assert mem['stage3']==70.; mem


## Problem 2

**Problem**: Explain why FSDP performs All-Gather before the forward pass.

### Solution

Each rank owns only a parameter shard, but dense layer computation needs the full weight. All-Gather temporarily reconstructs that layer’s full parameters before forward, then they are resharded after use.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
P,N=1000,4; local=P//N; gathered=local*N
assert gathered==P
{'local_shard':local,'required_for_layer_compute':gathered}


## Problem 3

**Problem**: Compare ZeRO-3 communication overhead with DDP.

### Solution

DDP ring all-reduce communicates roughly $2(N-1)P/N$ gradient elements. ZeRO-3 additionally all-gathers parameters for forward and backward, so it has more communication events and relies on bucket/prefetch overlap.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
P=1.; ddp=2*(1-1/8)*P; zero3_forward=P; zero3_backward=2*P
assert zero3_forward+zero3_backward>ddp
{'DDP_ring_units':ddp,'ZeRO3_approx_units':zero3_forward+zero3_backward}


## Problem 4

**Problem**: Explain why CPU offloading slows execution.

### Solution

Offload stores state in slower CPU DRAM instead of GPU HBM and adds PCIe/NVLink transfers to the critical path. The transfer lower bound is bytes/bandwidth, with small kernels and synchronization adding latency.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
bytes_to_move=8e9; pcie_bandwidth=32e9; transfer_seconds=bytes_to_move/pcie_bandwidth
assert transfer_seconds>.2; {'one_way_transfer_lower_bound_seconds':transfer_seconds}


## Problem 5

**Problem**: Calculate how many A100 80GB GPUs are needed to train a 70B model with ZeRO-3.

### Solution

A 70B model’s 16P states occupy about 1120 GB. Reserving 15% of each 80 GB GPU for activations/fragmentation leaves 68 GB, giving an ideal lower bound $\lceil1120/68\rceil=17$; real count requires profiling checkpointing and sequence length.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import math
parameters_billion=70; bytes_per_parameter=16; gpu_capacity_gb=80; headroom=.15
total_gb=parameters_billion*bytes_per_parameter
usable_per_gpu=gpu_capacity_gb*(1-headroom)
minimum=math.ceil(total_gb/usable_per_gpu)
feasible=lambda n: n*usable_per_gpu>=total_gb
enumerated=next(n for n in range(1,1000) if feasible(n))
assert minimum==enumerated and feasible(minimum) and not feasible(minimum-1)
{'model_state_gb':total_gb,'usable_gb_per_gpu':usable_per_gpu,'ideal_minimum':minimum,
 'boundary_check':{'previous_capacity_gb':(minimum-1)*usable_per_gpu,'minimum_capacity_gb':minimum*usable_per_gpu},
 'note':'ideal model-state bound only; measured activations and fragmentation can require more GPUs'}
